# Baseline — Cenário 3: CSV + PostgreSQL + REST API

**Spec equivalente:**  
> Leia `orders.csv`, `public.customers` e a API Open-Meteo (Recife, 7 dias). Faça join de orders + customers em `customer_id`. Adicione coluna `city_max_temp` com `max(temperature_2m_max)` para clientes em Recife; outros ficam nulo. Filtre `status == 'active'`. Salve em `public.enriched_orders`.

---

## Métricas de implementação

| Métrica | Valor |
|---|---|
| Tempo de implementação | ~18 min |
| Linhas de código | ver célula final |
| Decisões explícitas | 7 (url da API, qual campo de temp, qual agregação, tipo de join, chave do join, coluna de cidade, lógica de mapeamento) |
| Erros durante desenvolvimento | 2 (estrutura do JSON da API, type mismatch no join) |
| Iterações | 3 |

In [1]:
import os
import time
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(dotenv_path=Path("../../.env"))
POSTGRES_URL = os.getenv("POSTGRES_URL")
engine = create_engine(POSTGRES_URL)

start = time.time()

### 1. Extração

In [2]:
# CSV
orders = pd.read_csv("../data/orders.csv")
print(f"orders: {orders.shape}")

# PostgreSQL
with engine.connect() as conn:
    customers = pd.read_sql(text("SELECT * FROM public.customers"), conn)
print(f"customers: {customers.shape}")

# REST API — Open-Meteo Recife
url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=-8.05&longitude=-34.88"
    "&daily=temperature_2m_max"
    "&timezone=America/Recife&forecast_days=7"
)
resp = httpx.get(url, timeout=30)
resp.raise_for_status()
weather_raw = resp.json()
weather = pd.DataFrame(weather_raw["daily"])
print(f"weather: {weather.shape}")
weather.head()

orders: (10000, 7)
customers: (400, 4)


weather: (7, 2)


,time,temperature_2m_max
0,2026-06-23,27.0
1,2026-06-24,27.2
2,2026-06-25,28.3
3,2026-06-26,25.8
4,2026-06-27,25.9


### 2. Transformação

In [3]:
# Clean orders
orders = orders[orders["customer_id"].notnull()]
orders["customer_id"] = orders["customer_id"].astype(int)

# Join orders + customers
df = orders.merge(customers, on="customer_id", how="inner")

# Compute max temp from forecast (Recife only)
recife_max_temp = weather["temperature_2m_max"].max()

# Enrich: Recife customers get the temp, others get NaN
df["city_max_temp"] = df["city"].apply(
    lambda c: recife_max_temp if c == "Recife" else None
)

# Filter active
df = df[df["status"] == "active"].reset_index(drop=True)

print(f"After transform: {df.shape}")
print(f"city_max_temp non-null: {df['city_max_temp'].notnull().sum()}")
df.head(3)

After transform: (7135, 11)
city_max_temp non-null: 6232


,order_id,customer_id,order_dt,product,amount,qty,status,name,city,email,city_max_temp
0,11253,210,2024-09-17,Device Pro,785.21,15,active,Bruno Lima,Recife,customer210@example.com,28.3
1,9685,306,2024-07-14,Gadget Y,408.17,19,active,Carlos Santos,Olinda,customer306@example.com,NaN
2,6732,461,2024-03-13,Gadget X,54.00,21,active,Igor Santos,Recife,customer461@example.com,28.3


### 3. Qualidade (manual)

In [4]:
null_report = df.isnull().mean().round(4)
print("Null ratios (non-zero):")
print(null_report[null_report > 0])
print(f"\nDuplicates: {df.duplicated().sum()}")

Null ratios (non-zero):
city_max_temp    0.1266
dtype: float64

Duplicates: 134


### 4. Carga

In [5]:
df.to_sql("enriched_orders", engine, schema="public", if_exists="replace", index=False)

elapsed = time.time() - start
print(f"Saved {len(df)} rows → public.enriched_orders")
print(f"Pipeline execution time: {elapsed:.2f}s")

Saved 7135 rows → public.enriched_orders
Pipeline execution time: 1.71s


---
## Resumo de métricas


In [6]:
code_lines = 24

print("=" * 50)
print("BASELINE — Cenário 3")
print("=" * 50)
print(f"Rows output:              {len(df)}")
print(f"Execution time:           {elapsed:.2f}s")
print(f"LOC written:              {code_lines}")
print(f"Explicit decisions:       7")
print(f"Dev errors (tracebacks):  2")
print(f"Iterations to correct:    3")
print()
print("AI-ETL (avg of 5 runs):")
print(f"  Execution time:         24.5s")
print(f"  LOC written by human:   0 (spec in NL)")
print(f"  LLM attempts:           2.0")
print(f"  Human interventions:    0")

BASELINE — Cenário 3
Rows output:              7135
Execution time:           1.71s
LOC written:              24
Explicit decisions:       7
Dev errors (tracebacks):  2
Iterations to correct:    3

AI-ETL (avg of 5 runs):
  Execution time:         24.5s
  LOC written by human:   0 (spec in NL)
  LLM attempts:           2.0
  Human interventions:    0


---
## Tabela comparativa consolidada (3 cenários)

| Métrica | C1 Baseline | C1 AI-ETL | C2 Baseline | C2 AI-ETL | C3 Baseline | C3 AI-ETL |
|---|---|---|---|---|---|---|
| Tempo execução pipeline | <1s | 10.6s | <1s | 16.1s | <1s | 24.5s |
| Tempo de implementação | ~4 min | ~0 min (spec) | ~10 min | ~0 min (spec) | ~18 min | ~0 min (spec) |
| LOC escritas pelo humano | 9 | 0 | 16 | 0 | 24 | 0 |
| Decisões explícitas | 3 | 0 | 5 | 0 | 7 | 0 |
| Erros no desenvolvimento | 0 | — | 1 | — | 2 | — |
| Iterações | 1 | 2.0 (LLM) | 2 | 2.0 (LLM) | 3 | 2.0 (LLM) |
| Taxa de sucesso | 1/1 | 5/5 | 1/1 | 5/5 | 1/1 | 5/5 |

**Interpretação:**
- O pipeline manual é mais rápido na *execução* (sem latência de LLM), mas exige mais *tempo de implementação* e *decisões humanas* à medida que a complexidade aumenta.
- Para o Cenário 3, a implementação manual levou ~18 min e 3 iterações; o AI-ETL produziu resultado em ~25s sem nenhum código escrito pelo humano.
- O AI-ETL preserva auditabilidade: o código gerado é salvo em `{run_id}_transform.py` e pode ser inspecionado, diferentemente de um pipeline manual não versionado.
